# 공격 탐지 모델 선정 및 공통 컬럼 선택
기본 전처리 → 모델 비교 → CatBoost 중요도 계산 → 중요도 하위 공통 컬럼 삭제 → CatBoost 삭제 전후 F1 비교

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# 공격별 전처리 함수 불러오기
from process_attacks import (
    DUPLICATE_COLUMNS_TO_DROP,
    find_duplicate_column_pairs,
    load_data,
    preprocess_bruteforce,
    preprocess_dos,
)

# 모델 비교 및 컬럼 중요도 계산 함수 불러오기
from feature_selection_pipeline import (
    build_comparison_table,
    calculate_feature_importance,
    compare_models,
    create_selected_model,
    evaluate_model,
    find_common_low_importance_columns,
    select_best_model,
)

# 표 출력 형식 설정
pd.set_option("display.max_rows", None)
pd.set_option("display.float_format", lambda value: f"{value:.6f}")
pd.set_option("display.max_columns", None)

1. 기본 전처리 데이터 준비

In [2]:
# CSV 파일 경로 설정
BASE_DIR = Path.cwd()
BRUTE_CSV = BASE_DIR / "BruteForce.csv"
DOS_CSV = BASE_DIR / "DoS.csv"

# 공격별 데이터에서 전체 행의 값이 같은 중복 컬럼 확인
brute_raw = load_data(BRUTE_CSV)
dos_raw = load_data(DOS_CSV)
brute_duplicate_pairs = find_duplicate_column_pairs(brute_raw)
dos_duplicate_pairs = find_duplicate_column_pairs(dos_raw)
del brute_raw
del dos_raw

print("Brute Force 중복 컬럼 검사 결과")
display(brute_duplicate_pairs)

print("\nDoS 중복 컬럼 검사 결과")
display(dos_duplicate_pairs)

# 두 공격에서 동일하게 나타난 중복 컬럼 쌍 추출
common_duplicate_pairs = brute_duplicate_pairs.merge(
    dos_duplicate_pairs,
    on=["기준 컬럼", "중복 컬럼"],
    how="inner",
)
print("\n두 공격의 공통 중복 컬럼 검사 결과")
display(common_duplicate_pairs)

# 의미가 다른 TCP 플래그는 제외하고 안전한 중복 컬럼만 삭제
verified_duplicate_columns = set(
    common_duplicate_pairs["중복 컬럼"]
)
unverified_columns = set(DUPLICATE_COLUMNS_TO_DROP) - verified_duplicate_columns

print("\n모델 선정 후 삭제 실험에 사용할 공통 중복 컬럼")
display(pd.DataFrame({
    "삭제 컬럼": DUPLICATE_COLUMNS_TO_DROP
}))

# 공격별 데이터 전처리
X_brute, y_brute = preprocess_bruteforce(BRUTE_CSV)
X_dos, y_dos = preprocess_dos(DOS_CSV)

# 두 데이터에서 공통으로 존재하는 컬럼만 사용
common_features = X_brute.columns.intersection(X_dos.columns).tolist()
X_brute = X_brute[common_features].copy()
X_dos = X_dos[common_features].copy()

# 전처리 결과 확인
print(f"\nBrute Force: {X_brute.shape}, 공격 수: {int(y_brute.sum())}")
print(f"\nDoS: {X_dos.shape}, 공격 수: {int(y_dos.sum())}")
print(f"\n공통 입력 컬럼 수: {len(common_features)}")

Brute Force 중복 컬럼 검사 결과


,기준 컬럼,중복 컬럼
0,Total Fwd Packets,Subflow Fwd Packets
1,Total Backward Packets,Subflow Bwd Packets
2,Total Length of Fwd Packets,Subflow Fwd Bytes
3,Fwd Packet Length Mean,Avg Fwd Segment Size
4,Fwd PSH Flags,SYN Flag Count
5,Fwd Header Length,Fwd Header Length.1
6,RST Flag Count,ECE Flag Count



DoS 중복 컬럼 검사 결과


,기준 컬럼,중복 컬럼
0,Total Fwd Packets,Subflow Fwd Packets
1,Total Backward Packets,Subflow Bwd Packets
2,Total Length of Fwd Packets,Subflow Fwd Bytes
3,Fwd Packet Length Mean,Avg Fwd Segment Size
4,Fwd PSH Flags,SYN Flag Count
5,Fwd Header Length,Fwd Header Length.1



두 공격의 공통 중복 컬럼 검사 결과


,기준 컬럼,중복 컬럼
0,Total Fwd Packets,Subflow Fwd Packets
1,Total Backward Packets,Subflow Bwd Packets
2,Total Length of Fwd Packets,Subflow Fwd Bytes
3,Fwd Packet Length Mean,Avg Fwd Segment Size
4,Fwd PSH Flags,SYN Flag Count
5,Fwd Header Length,Fwd Header Length.1



모델 선정 후 삭제 실험에 사용할 공통 중복 컬럼


,삭제 컬럼
0,Subflow Fwd Packets
1,Subflow Bwd Packets
2,Subflow Fwd Bytes
3,Avg Fwd Segment Size
4,Fwd Header Length.1



Brute Force: (169558, 68), 공격 수: 1507

DoS: (691395, 68), 공격 수: 251712

공통 입력 컬럼 수: 68


2. 삭제 전 모델 비교 및 모델 선정

In [3]:
# 컬럼 삭제 전 모델별 F1 점수 계산
brute_before = compare_models(
    X_brute,
    y_brute,
    "Brute Force",
    include_svc=True,
)

# DoS의 파일 크기 때문에 SVC 모델 실행 시간이 매우 길어짐 =>  DoS에서만 제외
dos_before = compare_models(
    X_dos,
    y_dos,
    "DoS",
    include_svc=False,
)

# 두 공격의 평균 F1을 계산하여 최적 모델 선정
before_table = build_comparison_table(
    brute_before,
    dos_before,
    "삭제 전",
)
selected_model_name = select_best_model(before_table)

# 모델 선정 결과 출력
print(f"선정 모델: {selected_model_name}")
display(before_table.style.format({
    "Brute Force F1": "{:.6f}",
    "DoS F1": "{:.6f}",
    "Average F1": "{:.6f}",
}))

선정 모델: CatBoost


Dataset,Model,Brute Force F1,DoS F1,Average F1,Stage
0,CatBoost,0.994992,0.999732,0.997362,삭제 전
1,Extra Trees,0.988314,0.999057,0.993685,삭제 전
2,Random Forest,0.983108,0.999454,0.991281,삭제 전
3,Decision Tree,0.883549,0.955679,0.919614,삭제 전
4,Logistic Regression,0.892794,0.882250,0.887522,삭제 전
5,SVC,0.094637,nan,nan,삭제 전


3. 선정된 CatBoost로 두 데이터의 컬럼 중요도 계산

In [4]:
# 중요도와 삭제 전후 비교에 사용할 CatBoost 생성
importance_model_name = "CatBoost"
selected_model = create_selected_model(importance_model_name)

# 동일한 모델로 공격별 컬럼 중요도 계산
brute_importance = calculate_feature_importance(
    X_brute,
    y_brute,
    selected_model,
)
dos_importance = calculate_feature_importance(
    X_dos,
    y_dos,
    selected_model,
)

# Brute Force 중요도를 내림차순으로 출력
print("Brute Force 컬럼 중요도")
display(brute_importance.sort_values(
    by="Importance", ascending=False
).reset_index(drop=True))

# DoS 중요도를 내림차순으로 출력
print("DoS 컬럼 중요도")
display(dos_importance.sort_values(
    by="Importance", ascending=False
).reset_index(drop=True))

Brute Force 컬럼 중요도


,Feature,Importance
0,Destination Port,18.357938
1,Fwd IAT Min,9.810886
2,Bwd Packets/s,6.395836
3,act_data_pkt_fwd,6.356769
4,Flow IAT Min,4.848302
5,min_seg_size_forward,3.945851
6,Fwd IAT Std,3.265994
7,Fwd Packet Length Max,3.135669
8,Fwd Header Length,2.715922
9,Flow IAT Max,2.595344


DoS 컬럼 중요도


,Feature,Importance
0,Destination Port,33.797938
1,Bwd Packet Length Std,14.254925
2,Packet Length Mean,5.943288
3,Init_Win_bytes_backward,4.307416
4,Init_Win_bytes_forward,4.259098
5,Bwd Packets/s,4.174051
6,Min Packet Length,3.138663
7,Bwd Packet Length Min,2.355324
8,Flow IAT Min,2.304895
9,Flow Duration,1.776748


4. 중요도 0.1 이하 공통 컬럼 선정 및 삭제

In [5]:
# 두 공격에서 모두 중요도 0.1 이하인 컬럼 추출
common_drop_columns = find_common_low_importance_columns(
    brute_importance,
    dos_importance,
    threshold=0.1,
)
print(f"삭제 기준: 두 데이터 모두 중요도 <= 0.1")
print(f"공통 삭제 컬럼 수: {len(common_drop_columns)}")
display(pd.DataFrame({"공통 삭제 컬럼": common_drop_columns}))

# 공통 중복 컬럼과 공통 저중요도 컬럼을 하나의 삭제 목록으로 통합
all_drop_columns = sorted(
    set(DUPLICATE_COLUMNS_TO_DROP) | set(common_drop_columns)
)

# ===== 실제 두 데이터셋에 존재하는 컬럼만 남김 =====
all_drop_columns = [
    column
    for column in all_drop_columns
    if column in X_brute.columns and column in X_dos.columns
]

# 모델 선정 후 컬럼 삭제 실험 진행
X_brute_reduced = X_brute.drop(
    columns=all_drop_columns
).copy()

X_dos_reduced = X_dos.drop(
    columns=all_drop_columns
).copy()

y_brute_reduced = y_brute.copy()
y_dos_reduced = y_dos.copy()

# 삭제 후에도 두 데이터의 입력 컬럼 순서를 동일하게 정렬
remaining_features = X_brute_reduced.columns.intersection(
    X_dos_reduced.columns
).tolist()

X_brute_reduced = X_brute_reduced[remaining_features].copy()
X_dos_reduced = X_dos_reduced[remaining_features].copy()

print(f"삭제 후 공통 입력 컬럼 수: {len(remaining_features)}")

삭제 기준: 두 데이터 모두 중요도 <= 0.1
공통 삭제 컬럼 수: 8


,공통 삭제 컬럼
0,Active Max
1,Active Min
2,ECE Flag Count
3,Fwd PSH Flags
4,Idle Std
5,RST Flag Count
6,SYN Flag Count
7,Subflow Bwd Packets


삭제 후 공통 입력 컬럼 수: 56


5. 중복 / 저중요도 컬럼 삭제 후 CatBoost F1 비교

In [6]:
# CatBoost의 삭제 전 F1 가져오기
selected_before = before_table[
    before_table["Model"] == importance_model_name
].iloc[0]

# 중복 / 저중요도 컬럼 삭제 후 CatBoost F1 다시 계산
brute_after_f1 = evaluate_model(
    X_brute_reduced,
    y_brute_reduced,
    selected_model,
)
dos_after_f1 = evaluate_model(
    X_dos_reduced,
    y_dos_reduced,
    selected_model,
)

# 공격별 삭제 전후 F1 비교표 생성
catboost_f1_comparison = pd.DataFrame([
    {
        "Dataset": "Brute Force",
        "Model": importance_model_name,
        "삭제 전 F1": selected_before["Brute Force F1"],
        "삭제 후 F1": brute_after_f1,
    },
    {
        "Dataset": "DoS",
        "Model": importance_model_name,
        "삭제 전 F1": selected_before["DoS F1"],
        "삭제 후 F1": dos_after_f1,
    },
])
catboost_f1_comparison["F1 변화"] = (
    catboost_f1_comparison["삭제 후 F1"]
    - catboost_f1_comparison["삭제 전 F1"]
)

display(catboost_f1_comparison.style.format({
    "삭제 전 F1": "{:.6f}",
    "삭제 후 F1": "{:.6f}",
    "F1 변화": "{:+.6f}",
}))

,Dataset,Model,삭제 전 F1,삭제 후 F1,F1 변화
0,Brute Force,CatBoost,0.994992,0.991625,-0.003367
1,DoS,CatBoost,0.999732,0.999762,+0.000030


#### 이미 정제되어있던 연구용.csv을 대상으로 하다보니 오히려 colunm 삭제가 F1 score 하락 원인이 되는 것을 확인 가능하였다. 

#### 따라서 중요도가 낮더라도 전처리는 값이 비어있거나, 중복된 값만 colunms 제거를 마친 후 모델 학습 진행 예정이다.